In [3]:
import polars as pl
import os
import datetime

# ================= 配置区域 =================
DATA_ROOT = r"../QuantData/Ashare" 
CODE = "600941_SH" # 这里换成了招商银行，方便你验证
# ===========================================

def export_verification_data():
    print(f"🚀 开始生成通用版对账单: {CODE} ...")
    
    # -----------------------------------------------------------
    # 1. 加载数据 (保持不变)
    # -----------------------------------------------------------
    daily_path = os.path.join(DATA_ROOT, "stock_day_raw", f"{CODE}.parquet")
    df_daily = pl.read_parquet(daily_path).with_columns(
        pl.from_epoch("time", time_unit="ms").dt.replace_time_zone("UTC").dt.convert_time_zone("Asia/Shanghai").dt.date().alias("date")
    ).filter(pl.col("close").is_not_null()).sort("date")

    df_cap = pl.read_parquet(os.path.join(DATA_ROOT, "finance_capital", f"{CODE}.parquet")).with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("total_capital").cast(pl.Float64)
    ).sort("date")

    df_bal = pl.read_parquet(os.path.join(DATA_ROOT, "finance_balance", f"{CODE}.parquet")).with_columns(
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("date"),
        pl.col("tot_shrhldr_eqy_excl_min_int").cast(pl.Float64).alias("net_assets")
    ).sort("date")

    df_inc = pl.read_parquet(os.path.join(DATA_ROOT, "finance_income", f"{CODE}.parquet")).with_columns([
        pl.col("m_anntime").str.strptime(pl.Date, "%Y%m%d").alias("pub_date"),
        pl.col("m_timetag").str.strptime(pl.Date, "%Y%m%d").alias("report_date"),
        pl.col("net_profit_excl_min_int_inc").cast(pl.Float64).alias("cum_profit")
    ]).sort("pub_date")

    df_div = pl.read_parquet(os.path.join(DATA_ROOT, "finance_dividend", f"{CODE}.parquet")).with_columns(
        pl.col("date").str.strptime(pl.Date, "%Y%m%d"),
        pl.col("interest").cast(pl.Float64)
    ).group_by("date").agg(pl.col("interest").sum()).sort("date")

    # -----------------------------------------------------------
    # 2. 【核心新增】计算历史平均分红率 (Self-Adaptive Payout Ratio)
    # -----------------------------------------------------------
    print("🤖 正在学习该股票的历史分红习惯...")
    
    # A. 整理每年的净利润 (按 report_date 的年份归集 Q4 数据)
    df_yearly_profit = df_inc.with_columns(
        pl.col("report_date").dt.year().alias("year"),
        pl.col("report_date").dt.month().alias("month")
    ).filter(pl.col("month") == 12).group_by("year").last().select(["year", "cum_profit"]).sort("year")
    
    # B. 整理每年的分红总额 (按除权日 date 的年份归集)
    # 注意：2024年的分红(interest)通常来自于2023年的利润
    # 所以我们将 分红年份 - 1 = 对应的利润年份
    df_yearly_div = df_div.with_columns(
        (pl.col("date").dt.year() - 1).alias("profit_year") # 关键位移
    ).group_by("profit_year").agg(pl.col("interest").sum().alias("total_div")).sort("profit_year")

    # C. 合并计算分红率
    # 我们需要当时的总股本才能算出总分红金额，或者我们直接对比：每股分红 / 每股收益(EPS)
    # 为了简化且准确，我们用 (每股分红 * 股本) / 总利润 ? 不，直接用 Total Dividends / Total Profit 会有股本变动干扰
    # 最准的方法：每股分红 / (归母净利润 / 当时总股本)
    
    # 这里我们采用简化算法：假设股本变动不大，直接计算 Ratio
    # 先把股本附上去(用年末股本)
    df_yearly_cap = df_cap.with_columns(
        pl.col("date").dt.year().alias("year")
    ).group_by("year").last().select(["year", "total_capital"])

    df_payout_history = df_yearly_profit.join(df_yearly_div, left_on="year", right_on="profit_year", how="inner")
    df_payout_history = df_payout_history.join(df_yearly_cap, on="year", how="left")
    
    # 计算实际分红率 = (每股分红 * 总股本) / 归母净利润
    df_payout_history = df_payout_history.with_columns(
        (pl.col("total_div") * pl.col("total_capital") / pl.col("cum_profit")).alias("payout_ratio")
    ).filter(
        (pl.col("payout_ratio") > 0) & (pl.col("payout_ratio") < 1.2) # 过滤异常值
    )
    
    # 取最近 3 年的平均值
    if df_payout_history.height >= 1:
        avg_payout_ratio = df_payout_history.tail(3)["payout_ratio"].mean()
        print(f"✅ 历史分红率计算成功 (参考最近{df_payout_history.height}年): {avg_payout_ratio*100:.2f}%")
        print("   详细记录:", df_payout_history.tail(3).select(["year", "payout_ratio"]).to_dicts())
    else:
        avg_payout_ratio = 0.30 # 兜底默认值
        print("⚠️ 无法计算历史分红率，使用默认值 30%")

    # -----------------------------------------------------------
    # 3. 常规 TTM 计算 (保持不变)
    # -----------------------------------------------------------
    min_date = df_daily["date"].min()
    max_date = df_daily["date"].max()
    df_calendar = pl.date_range(min_date, max_date, "1d", eager=True).to_frame("date")
    
    df_div_ttm = df_calendar.join(df_div, on="date", how="left").with_columns(
        pl.col("interest").fill_null(0.0)
    ).with_columns(
        pl.col("interest").rolling_sum(window_size=365).alias("div_ttm")
    )
    
    df_inc_fix = df_inc.with_columns(
        pl.col("report_date").dt.month().alias("rpt_month")
    ).with_columns(
        pl.when(pl.col("rpt_month") == 3).then(4.0)
          .when(pl.col("rpt_month") == 6).then(2.0)
          .when(pl.col("rpt_month") == 9).then(1.33333333)
          .otherwise(1.0)
          .alias("annual_factor")
    )
    df_inc_simple = df_inc_fix.group_by("pub_date").last().rename({"pub_date": "date"}).sort("date")

    # -----------------------------------------------------------
    # 4. 合并与导出
    # -----------------------------------------------------------
    df_main = df_daily.join_asof(df_cap, on="date", strategy="backward")
    df_main = df_main.join_asof(df_bal, on="date", strategy="backward")
    df_main = df_main.join_asof(df_inc_simple.select(["date", "cum_profit", "annual_factor"]), on="date", strategy="backward")
    df_main = df_main.with_columns((pl.col("cum_profit") * pl.col("annual_factor")).alias("net_profit_ttm_approx"))
    df_main = df_main.join(df_div_ttm.select(["date", "div_ttm"]), on="date", how="left")

    df_final = df_main.with_columns([
        (pl.col("close") * pl.col("total_capital")).alias("market_cap")
    ]).with_columns([
        (pl.col("market_cap") / pl.col("net_assets")).alias("PB"),
        (pl.col("market_cap") / pl.col("net_profit_ttm_approx")).alias("PE"),
        (pl.col("div_ttm") / pl.col("close") * 100).alias("Yield_Pct")
    ]).select([
        "date", "close", "total_capital", "net_assets", "net_profit_ttm_approx", "div_ttm", "PB", "PE", "Yield_Pct"
    ]).drop_nulls()

    df_final.write_csv("check_valuation.csv")

    # -----------------------------------------------------------
    # 5. 采样打印 (含自适应分红预测)
    # -----------------------------------------------------------
    print("\n🔎 历史数据抽样核对:")
    if df_final.height > 0:
        idx_now = df_final.height - 1
        row = df_final[idx_now]
        curr_date = row["date"][0]
        
        close_price = row['close'][0]
        total_shares = row['total_capital'][0]
        ttm_profit = row['net_profit_ttm_approx'][0]
        
        # --- 前瞻预测 (使用自适应分红率) ---
        print(f"\n📅 采样日期: {curr_date}")
        print(f"   [基础估值]")
        print(f"   收盘价:      {close_price:.2f}")
        print(f"   PB:          {row['PB'][0]:.2f}")
        print(f"   PE:          {row['PE'][0]:.2f}")
        print(f"   股息率(TTM): {row['Yield_Pct'][0]:.2f}%")
        
        print(f"\n   [前瞻预测 - 自适应模式]")
        
        latest_report = df_inc.filter(pl.col("pub_date") <= curr_date).tail(1)
        if latest_report.height > 0:
            rpt_profit = latest_report["cum_profit"][0]
            rpt_month = latest_report["report_date"][0].month
            
            # 1. 预估全年利润 (线性外推)
            if rpt_month == 9: est_full_year_profit = rpt_profit / 3 * 4
            elif rpt_month == 6: est_full_year_profit = rpt_profit * 2
            elif rpt_month == 12: est_full_year_profit = rpt_profit
            else: est_full_year_profit = ttm_profit
            
            # 2. 【关键】使用计算出的历史平均分红率
            payout_ratio_est = avg_payout_ratio 
            
            est_total_div_amt = est_full_year_profit * payout_ratio_est
            est_eps_div = est_total_div_amt / total_shares
            forward_yield = (est_eps_div / close_price) * 100
            
            print(f"   预估全利润:  {est_full_year_profit/1e8:.2f} 亿元")
            print(f"   历史分红率:  {payout_ratio_est*100:.2f}% (自动计算)")
            print(f"   预估全分红:  {est_eps_div:.2f} 元/股")
            print(f"   ★ 前瞻股息率: {forward_yield:.2f}%")
            
            # --- 敏感性矩阵 (自动调整区间) ---
            print(f"\n   [敏感性矩阵 (基准分红率 {payout_ratio_est*100:.0f}%)]")
            # 自动生成围绕基准分红率的区间
            base_p = int(payout_ratio_est * 100)
            payout_scenarios = [base_p-2, base_p, base_p+2, base_p+5]
            growth_scenarios = [-0.02, 0.00, 0.03, 0.05, 0.08]

            print(f"   {'-'*65}")
            header = f"   {'利润增长':<10} |"
            for p in payout_scenarios: header += f" {'分红'+str(p)+'%':<10} |"
            print(header)
            print(f"   {'-'*65}")

            for g in growth_scenarios:
                row_str = f"   {g*100:>+6.0f}%    |"
                for p in payout_scenarios:
                    profit_2026 = est_full_year_profit * (1 + g)
                    div_2026 = (profit_2026 * (p/100.0)) / total_shares
                    yield_2026 = (div_2026 / close_price) * 100
                    
                    val_str = f"{yield_2026:.2f}%"
                    if yield_2026 >= 6.0: val_str += "🔥"
                    elif yield_2026 >= 5.0: val_str += "✅"
                    else: val_str += "  "
                    row_str += f" {val_str:<10} |"
                print(row_str)
            print(f"   {'-'*65}")
            
        else:
            print("   (无财报数据，无法预测)")

if __name__ == "__main__":
    export_verification_data()

🚀 开始生成通用版对账单: 600941_SH ...
🤖 正在学习该股票的历史分红习惯...
✅ 历史分红率计算成功 (参考最近3年): 73.41%
   详细记录: [{'year': 2022, 'payout_ratio': 0.7158970864097848}, {'year': 2023, 'payout_ratio': 0.7408285084577174}, {'year': 2024, 'payout_ratio': 0.7454934960196116}]

🔎 历史数据抽样核对:

📅 采样日期: 2025-12-17
   [基础估值]
   收盘价:      102.31
   PB:          1.62
   PE:          14.40
   股息率(TTM): 4.69%

   [前瞻预测 - 自适应模式]
   预估全利润:  1538.04 亿元
   历史分红率:  73.41% (自动计算)
   预估全分红:  5.22 元/股
   ★ 前瞻股息率: 5.10%

   [敏感性矩阵 (基准分红率 73%)]
   -----------------------------------------------------------------
   利润增长       | 分红71%      | 分红73%      | 分红75%      | 分红78%      |
   -----------------------------------------------------------------
       -2%    | 4.83%      | 4.97%      | 5.11%✅     | 5.31%✅     |
       +0%    | 4.93%      | 5.07%✅     | 5.21%✅     | 5.42%✅     |
       +3%    | 5.08%✅     | 5.22%✅     | 5.37%✅     | 5.58%✅     |
       +5%    | 5.18%✅     | 5.32%✅     | 5.47%✅     | 5.69%✅     |
       +8%    | 5.33%✅    